# Model 2: TF-IDF n-grams + Logistic Regression, with data augmentation

Changes with respect to Model 1:
* word uni- and bigrams (`ngram_range=(1, 2)`), to capture terms such as "blood pressure" or "no fracture";
* Logistic Regression instead of a linear SVM;
* data augmentation of the minority classes, applied to the training split only.

All variants use the same stratified 80/20 split as Models 1 and 3 (`random_state=42`) and are evaluated with the two corpus rules of the final system:
* **keyword rule:** if the first keyword names a specialty, that specialty is returned;
* **duplicate-note rule:** if a record's transcription appears in the training data with label A, A is excluded.

## 1. Data, split and shared helpers

In [6]:
import random, re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
train = pd.read_csv("train_clean.csv", sep=";", dtype=str, keep_default_na=False)
LABELS = sorted(train["medical_specialty"].unique())

def keyword_specialty(keywords):
    first = keywords.split(",")[0].strip().lower().replace(" / ", "-").replace("/", "-")
    return {l.lower(): l for l in LABELS}.get(first)

train["rule"] = train["keywords"].apply(keyword_specialty)                 # keyword rule (None if no keyword)
train["keywords"] = train["keywords"].apply(lambda k: ",".join(k.split(",")[1:]))   # text models never see it
train["text"] = train["description"] + " " + train["sample_name"] + " " + train["keywords"] + " " + train["transcription"]

idx_tr, idx_te = train_test_split(train.index, test_size=0.2, stratify=train["medical_specialty"], random_state=SEED)
tr, te = train.loc[idx_tr], train.loc[idx_te]
no_kw = te["rule"].isna().to_numpy()
print(len(tr), "train |", len(te), "test |", no_kw.sum(), "test notes without keywords")

def exclude_seen(scores, classes, df_test, df_train):
    # duplicate-note rule: a note already in training cannot get a label it has there
    seen = df_train[df_train.transcription.str.strip() != ""].groupby("transcription")["medical_specialty"].agg(set)
    s = np.array(scores, dtype=float); cl = list(classes)
    for i, note in enumerate(df_test["transcription"]):
        for l in seen.get(note, set()):
            s[i, cl.index(l)] = -np.inf
    return np.array(cl)[s.argmax(1)]

def scores_of(model, X):
    return model.decision_function(X) if hasattr(model, "decision_function") else np.log(model.predict_proba(X) + 1e-12)

def evaluate_layers(name, model, df_fit, X_fit, X_test):
    # fit, then score the 3 layers: plain -> + duplicate rule -> + keyword rule, plus the no-keyword notes
    model.fit(X_fit, df_fit["medical_specialty"])
    S = scores_of(model, X_test)
    plain = np.array(model.classes_)[S.argmax(1)]
    dup = exclude_seen(S, model.classes_, te, tr)          # duplicates are checked against the real training part
    full = np.where(te["rule"].notna(), te["rule"], dup)
    y = te["medical_specialty"].to_numpy()
    row = {"model": name}
    for tag, p, yy in [("plain", plain, y), ("+dup", dup, y), ("full", full, y), ("no-KW", dup[no_kw], y[no_kw])]:
        f = f1_score(yy, p, labels=LABELS, average=None, zero_division=0)
        row[f"{tag} acc"] = accuracy_score(yy, p); row[f"{tag} mF1"] = f.mean()
        if tag in ("plain", "full"): row[f"{tag} minF1"] = f.min()
    return row

2135 train | 534 test | 94 test notes without keywords


## 2. Data augmentation (training split only)
Two strategies from Easy Data Augmentation , applied to every class with fewer than 100 training records (Dermatology, Psychiatry-Psychology, Ophthalmology, Neurosurgery). Each original record receives 3 augmented copies:
* **Random swap:** 3 random pairs of words are swapped;
* **Swap + deletion:** each word is deleted with probability 0.1, then 5% of the word positions are swapped.

Synonym replacement is not used, since general-purpose thesauri do not cover clinical vocabulary. The test split is never augmented.

In [7]:
def random_swap(text, rng, n=3):
    w = text.split()
    if len(w) < 4: return text
    for _ in range(n):
        i, j = rng.sample(range(len(w)), 2); w[i], w[j] = w[j], w[i]
    return " ".join(w)

def eda(text, rng, p_del=0.1, swap_rate=0.05):
    w = [x for x in text.split() if rng.random() > p_del] or text.split()
    for _ in range(max(1, int(swap_rate * len(w)))):
        i, j = rng.randrange(len(w)), rng.randrange(len(w)); w[i], w[j] = w[j], w[i]
    return " ".join(w)

def augment(df, method, seed, threshold=100, copies=3):
    rng = random.Random(seed)
    counts = df["medical_specialty"].value_counts()
    minority = counts[counts < threshold].index
    rows = [{"medical_specialty": r.medical_specialty, "text": method(r.text, rng)}
            for r in df[df.medical_specialty.isin(minority)].itertuples() for _ in range(copies)]
    return pd.concat([df[["medical_specialty", "text"]], pd.DataFrame(rows)], ignore_index=True)

print("minority classes:", list(tr.medical_specialty.value_counts().loc[lambda s: s < 100].index))
print("training notes before / after augmentation:", len(tr), "/", len(augment(tr, random_swap, 0)))

minority classes: ['Neurosurgery', 'Ophthalmology', 'Psychiatry-Psychology', 'Dermatology']
training notes before / after augmentation: 2135 / 2657


## 3. Models compared on the same split
* **Baseline:** Multinomial Naive Bayes on TF-IDF features of the description, lower-cased and Snowball-stemmed;
* **Model 1:** TF-IDF unigrams + linear SVM (`C=0.3`, balanced);
* **Model 2:** TF-IDF uni- and bigrams (`min_df=2`, sublinear TF) + Logistic Regression (`C=10`, balanced), without augmentation, with random swap, and with swap + deletion. Augmented variants are averaged over 3 augmentation seeds;
* **Model 2 with mode de-duplication:** each repeated text is kept once with its most frequent label (alphabetical order on ties).

In [8]:
from nltk.stem.snowball import SnowballStemmer
stem = SnowballStemmer("english"); tok = re.compile(r"[a-z]+")
stem_tok = lambda t: [stem.stem(w) for w in tok.findall(t.lower())]

def M2(): return make_pipeline(TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2)),
                               LogisticRegression(C=10, class_weight="balanced", max_iter=3000))
rows = []
rows.append(evaluate_layers("Baseline (MNB, description)",
    make_pipeline(TfidfVectorizer(tokenizer=stem_tok, token_pattern=None, lowercase=False), MultinomialNB()),
    tr, tr["description"], te["description"]))
rows.append(evaluate_layers("Model 1: TF-IDF + SVM",
    make_pipeline(TfidfVectorizer(sublinear_tf=True, min_df=2), LinearSVC(C=0.3, class_weight="balanced")),
    tr, tr["text"], te["text"]))
rows.append(evaluate_layers("Model 2: n-grams + LR", M2(), tr, tr["text"], te["text"]))
for name, method in [("Model 2 + random swap", random_swap), ("Model 2 + EDA", eda)]:
    runs = []
    for s in (0, 1, 2):
        aug = augment(tr, method, seed=s)
        runs.append(evaluate_layers(name, M2(), aug, aug["text"], te["text"]))
    r = pd.DataFrame(runs); mean = r.drop(columns="model").mean().to_dict(); mean["model"] = name
    mean["no-KW acc std"] = r["no-KW acc"].std(); rows.append(mean)
dedup = tr.groupby("text")["medical_specialty"].agg(lambda x: x.mode()[0]).reset_index()
rows.append(evaluate_layers("Model 2 + mode de-duplication", M2(), dedup, dedup["text"], te["text"]))

table = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 250)
display((table * 100).round(1))
(table * 100).round(1).to_csv("figures/models_comparison.csv")

,plain acc,plain mF1,plain minF1,+dup acc,+dup mF1,full acc,full mF1,full minF1,no-KW acc,no-KW mF1,no-KW acc std
model,,,,,,,,,,,
"Baseline (MNB, description)",39.3,13.8,0.0,54.5,34.7,88.2,84.5,22.2,33.0,31.0,NaN
Model 1: TF-IDF + SVM,44.2,49.0,20.3,76.2,76.3,94.6,93.1,79.2,69.1,62.1,NaN
Model 2: n-grams + LR,40.1,42.9,18.2,77.3,76.2,94.8,93.3,79.6,70.2,63.3,NaN
Model 2 + random swap,38.0,40.3,16.6,77.4,76.1,94.8,93.3,79.6,70.2,63.3,0.0
Model 2 + EDA,37.7,40.0,15.8,77.3,76.0,94.8,93.4,79.4,70.2,63.4,0.0
Model 2 + mode de-duplication,38.6,42.6,11.1,76.6,77.0,94.8,93.3,80.4,70.2,62.7,NaN


## 4. Augmentation before vs. after the split
If augmentation is applied before splitting, near-identical copies of a record fall in both splits and the model is evaluated on records it has effectively seen. Both orders are compared below on the de-duplicated data.

In [9]:
base = train.groupby("text")["medical_specialty"].agg(lambda x: x.mode()[0]).reset_index()
m = M2()
# (a) augment, then split (leaky)
a = augment(base, random_swap, 0)
a_tr, a_te = train_test_split(a, test_size=0.2, stratify=a["medical_specialty"], random_state=SEED)
p = m.fit(a_tr["text"], a_tr["medical_specialty"]).predict(a_te["text"])
f = f1_score(a_te["medical_specialty"], p, average=None)
print(f"augment BEFORE split : acc={accuracy_score(a_te['medical_specialty'], p):.3f}  minF1={f.min():.3f}")
# (b) split, then augment the training part only
b_tr, b_te = train_test_split(base, test_size=0.2, stratify=base["medical_specialty"], random_state=SEED)
b = augment(b_tr, random_swap, 0)
p = m.fit(b["text"], b["medical_specialty"]).predict(b_te["text"])
f = f1_score(b_te["medical_specialty"], p, average=None)
print(f"augment AFTER split  : acc={accuracy_score(b_te['medical_specialty'], p):.3f}  minF1={f.min():.3f}")
print("\nmode de-duplication keeps 'Surgery' in", (base.merge(train[['text']].assign(n=1).groupby('text').n.sum().reset_index(), on='text')
      .query('n > 1').medical_specialty == 'Surgery').sum(), "of the repeated texts")

augment BEFORE split : acc=0.841  minF1=0.692
augment AFTER split  : acc=0.773  minF1=0.333

mode de-duplication keeps 'Surgery' in 0 of the repeated texts


## 5. Observations
* For Models 1 and 2, the duplicate-note rule adds 32 to 37 points of accuracy and the keyword rule a further 17 to 19, more than any change of text model.
* On the records without keywords (*no-KW*), the three Model 2 variants with and without augmentation obtain the same accuracy (70.2%).
* Mode de-duplication never keeps the label Surgery for repeated texts (alphabetical tie-break), which lowers the minimum per-class F1 without rules.